# ANLP — Session 2, Part 1: Linguistic feature extraction

**Hands-on notebook**

- POS tagging
- chunking
- dependency parsing

## Setup

In [1]:
%%capture
import sys

# Install dependencies when running in Google Colab.
# On your own computer, set up the environment as described in README.md instead.
if "google.colab" in sys.modules:
    print("Google Colab environment detected. Installing dependencies...")

    # install
    %pip install spacy==3.8.14
    !python -m spacy download en_core_web_sm-3.8.0

## Load the model and the corpus

In [2]:
from pprint import pprint
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Load our working corpus: "Alice's Adventures in Wonderland" (public domain, Project Gutenberg)
# Falls back to a small built-in sample if you are offline.
import urllib.request

FALLBACK = (
    "Alice was beginning to get very tired of sitting by her sister on the bank, "
    "and of having nothing to do: once or twice she had peeped into the book her sister "
    "was reading, but it had no pictures or conversations in it, 'and what is the use of "
    "a book,' thought Alice 'without pictures or conversations?' "
    "So she was considering in her own mind (as well as she could, for the hot day made her "
    "feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth "
    "the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink "
    "eyes ran close by her. There was nothing so very remarkable in that; nor did Alice think it "
    "so very much out of the way to hear the Rabbit say to itself, 'Oh dear! Oh dear! I shall be late!'"
)

try:
    url = "https://raw.githubusercontent.com/liri-uzh/ANLP-HS2026/main/data/alice_in_wonderland.txt"   # copy of the Gutenberg text, stored in the course repo
    raw = urllib.request.urlopen(url, timeout=10).read().decode("utf-8")
    # strip the Gutenberg header/footer
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1
    end = raw.find("*** END OF")
    text = raw[start:end].strip()
    print(f"Downloaded corpus: {len(text):,} characters")
except Exception as e:
    text = FALLBACK
    print(f"Download failed ({e}); using built-in sample: {len(text):,} characters")

In [4]:
doc = nlp(text)
print(f"{len(doc):,} tokens ready")

37,713 tokens ready


## Part-of-speech tagging

In [5]:
sample = nlp("The White Rabbit checked its pocket watch and ran quickly down the hole.")
for t in sample:
    print(f"{t.text:>10}  {t.pos_:>6}  {spacy.explain(t.pos_)}")

       The     DET  determiner
     White   PROPN  proper noun
    Rabbit   PROPN  proper noun
   checked    VERB  verb
       its    PRON  pronoun
    pocket    NOUN  noun
     watch    NOUN  noun
       and   CCONJ  coordinating conjunction
       ran    VERB  verb
   quickly     ADV  adverb
      down     ADP  adposition
       the     DET  determiner
      hole    NOUN  noun
         .   PUNCT  punctuation


In [6]:
dir(t)

['_',
 '__bytes__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__ne__',
 '__new__',
 '__pyx_vtable__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__unicode__',
 'ancestors',
 'check_flag',
 'children',
 'cluster',
 'conjuncts',
 'dep',
 'dep_',
 'doc',
 'ent_id',
 'ent_id_',
 'ent_iob',
 'ent_iob_',
 'ent_kb_id',
 'ent_kb_id_',
 'ent_type',
 'ent_type_',
 'get_extension',
 'has_dep',
 'has_extension',
 'has_head',
 'has_morph',
 'has_vector',
 'head',
 'i',
 'idx',
 'iob_strings',
 'is_alpha',
 'is_ancestor',
 'is_ascii',
 'is_bracket',
 'is_currency',
 'is_digit',
 'is_left_punct',
 'is_lower',
 'is_oov',
 'is_punct',
 'is_quote',
 'is_right_punct',
 'is_sent_end',
 'is_sent_start',
 'is_space',
 'is_stop',
 'is_title',
 'is_upper',
 'lang

In [7]:
for t in sample:
    print(f"{t.text:>10}  {t.pos_:>6}   {t.tag_:>6}")

       The     DET       DT
     White   PROPN      NNP
    Rabbit   PROPN      NNP
   checked    VERB      VBD
       its    PRON     PRP$
    pocket    NOUN       NN
     watch    NOUN       NN
       and   CCONJ       CC
       ran    VERB      VBD
   quickly     ADV       RB
      down     ADP       IN
       the     DET       DT
      hole    NOUN       NN
         .   PUNCT        .


### Comprehension Questions:

1. Which tag set is used when calling `.pos_` on a token object?

2. How can we inspect POS with an alternative, more granular tag set?

3. What is the advantage of each of these types of tag sets?

In [8]:
# Which parts of speech dominate the corpus?
from collections import Counter
pos_counts = Counter(t.pos_ for t in doc if not (t.is_space or t.pos_ == 'PUNCT'))
pos_counts.most_common(8)

[('VERB', 4501),
 ('PRON', 4246),
 ('NOUN', 3571),
 ('ADP', 2757),
 ('DET', 2757),
 ('ADV', 2073),
 ('AUX', 1788),
 ('PROPN', 1523)]

👉 **Your turn** — write code to identify the most fequently occuring verbs in the corpus

In [9]:
# your code here...

In [10]:
verb_counts = Counter(t.text for t in doc if t.pos_ == 'VERB')
verb_counts.most_common(10)


[('said', 458),
 ('’s', 109),
 ('know', 87),
 ('went', 82),
 ('thought', 70),
 ('see', 66),
 ('began', 58),
 ('do', 53),
 ('had', 53),
 ('think', 53)]

## Chunking

Noun chunks can provide quick insights and often extract the phrases you actually care about.

Importantly, we apply some simple filtering here to discard chunks whose head is a stop word. E.g. we don't want noun chunks consisting of 'it', 'they', 'she', etc.

In [11]:
sample = nlp("The White Rabbit checked its pocket watch and ran quickly down the hole.")
for chunk in sample.noun_chunks:
    print(f"{chunk.text.lower()}")

the white rabbit
its pocket watch
the hole


In [12]:
chunk_counts = Counter(
    chunk.text.lower() for chunk in doc.noun_chunks
    if not chunk.root.is_stop  # drop chunks headed by a stopword ("it", "she", ...)
)
chunk_counts.most_common(15)

[('alice', 361),
 ('_', 221),
 ('the queen', 58),
 ('the king', 58),
 ('the hatter', 50),
 ('the gryphon', 49),
 ('the mock turtle', 48),
 ('the duchess', 34),
 ('the dormouse', 34),
 ('the march hare', 27),
 ('the mouse', 24),
 ('the caterpillar', 23),
 ('the cat', 22),
 ('the door', 19),
 ('the white rabbit', 18)]

#### Comprehension Questions

1. How do these phrases compare to n-grams?

2. With this method, we only get NOUN chunks. What are the limitations? Where could it be useful to have more detailed phrasal information?

## Dependency Parsing

In [13]:
sample = nlp("The White Rabbit checked its pocket watch and ran quickly down the hole.")

print("Token       Dependency     Head Token     Dependency Type")
print("-----------------------------------------------------------")
for t in sample:
    print(f"{t.text:<10}  {t.dep_:<14} {t.head.text:<12} {spacy.explain(t.dep_)}")

Token       Dependency     Head Token     Dependency Type
-----------------------------------------------------------
The         det            Rabbit       determiner
White       compound       Rabbit       compound
Rabbit      nsubj          checked      nominal subject
checked     ROOT           checked      root
its         poss           watch        possession modifier
pocket      compound       watch        compound
watch       dobj           checked      direct object
and         cc             checked      coordinating conjunction
ran         conj           checked      conjunct
quickly     advmod         ran          adverbial modifier
down        prep           ran          prepositional modifier
the         det            hole         determiner
hole        pobj           down         object of preposition
.           punct          checked      punctuation


In [14]:
# Visualise the dependency tree
from spacy import displacy

displacy.render(sample, style="dep")

In [15]:
def extract_basic_svo_triples(doc: spacy.tokens.Doc) -> list[tuple]:
    """
    Extracts Subject-Verb-Object triples from a spaCy Doc.
    """
    triples = []

    # Iterate over the tokens
    for token in doc:

        # Check if the token is a verb
        if token.pos_ == "VERB":
            # Initialise the subject and obj for the current verb
            subj = None
            obj = None

            # Find subject and object among children
            for child in token.children:
                if child.dep_ == "nsubj" or child.dep_ == "nsubjpass":
                    subj = child.text
                elif child.dep_ == "dobj":
                    obj = child.text

            # If we found a subject and a direct object, add the triple
            if subj and obj:
                triples.append((subj, token.text, obj))

    return triples

In [16]:
# Example usage:
svo_triples = extract_basic_svo_triples(sample)

print("Subject-Verb-Object Triples:")
for s, v, o in svo_triples:
    print(f"Subject: {s}, Verb: {v}, Object: {o}")

Subject-Verb-Object Triples:
Subject: Rabbit, Verb: checked, Object: watch


In [17]:
sentence_passive = nlp("The cake was eaten by the boy.")
svo_triples_passive = extract_basic_svo_triples(sentence_passive)

print("\nSubject-Verb-Object Triples (Passive Voice):")
for s, v, o in svo_triples_passive:
    print(f"Subject: {s}, Verb: {v}, Object: {o}")


Subject-Verb-Object Triples (Passive Voice):


### Comprehension Questions

1. Why does the `extract_basic_svo_triples()` function not return anything on the passive voice sentence? Hint: inspect the dependency tags for the passive voice sentence...


In [18]:
# your code here ...

In [19]:
for t in sentence_passive:
    print(f"{t.text:<10}  {t.dep_:<14} {t.head.text:<12} {spacy.explain(t.dep_)}")

The         det            cake         determiner
cake        nsubjpass      eaten        nominal subject (passive)
was         auxpass        eaten        auxiliary (passive)
eaten       ROOT           eaten        root
by          agent          eaten        agent
the         det            boy          determiner
boy         pobj           by           object of preposition
.           punct          eaten        punctuation


👉 **Your turn** — write a function to find all adjectives modifying a specific noun in the corpus (or subcorpus).

In [20]:
def find_adjectives_modifying_noun(target_noun: str, corpus: spacy.tokens.Doc) -> list[str]:
    """
    Finds all adjectives modifying a specific target noun in the corpus.

    Note, if a word's `dep_` attribute is `amod`, it means it's an adjective modifying its head noun.

    Logic to apply
        1. Iterate through each token in the corpus, and for each token:
        2. Check if its dependency type (`token.dep_`) is 'amod' (adjectival modifier)
        3. If it is an 'amod', check if its head token's text (`token.head.text`) matches the `target_noun` (case-insensitive for robustness, or as specified)
        4. If both conditions are met, add the token's text (`token.text`) to the `adjectives` list
    """
    adjectives = []

    # your code here...

    return adjectives

In [21]:
def find_adjectives_modifying_noun(target_noun: str, corpus: spacy.tokens.Doc) -> list[str]:
    """
    Finds all adjectives modifying a specific target noun in the corpus.

    Note, if a word's `dep_` attribute is `amod`, it means it's an adjective modifying its head noun.

    Logic to apply
        1. Iterate through each token in the corpus, and for each token:
        2. Check if its dependency type (`token.dep_`) is 'amod' (adjectival modifier)
        3. If it is an 'amod', check if its head token's text (`token.head.text`) matches the `target_noun` (case-insensitive for robustness, or as specified)
        4. If both conditions are met, add the token's text (`token.text`) to the `adjectives` list
    """
    adjectives = []

    for token in corpus:
        if token.dep_ == 'amod' and token.head.text.lower() == target_noun.lower():
            adjectives.append(token.text)

    return adjectives


In [22]:
n = 'Alice'
adjs = Counter(find_adjectives_modifying_noun(n, doc))
adjs

Counter({'poor': 10,
         'little': 3,
         'confused': 2,
         'wise': 1,
         'glad': 1,
         'noticed': 1,
         'foolish': 1,
         'alarmed': 1,
         'round': 1})